# **Phase 4: Fuel-Typology Clustering**
---
**Research question:** Can Australian biomass and waste fuels be grouped into
physically meaningful clusters based on energy content and thermochemical
behavior, independent of administrative labels?

This tests whether measured fuel properties align with the recorded
`Biomass_Type`, supports downstream technology screening (combustion /
gasification / pyrolysis), and requires no target variable (unsupervised).

#### **Phase 4 is grouped into 4 stages:**
---
**4:** Custering (KMeans)\
**4A:** Process suitability rules\
**4B:** Cluster explainability (feature importance)\
**4C:** WtE conversion rules (cluster + process suitability)  

#### **Selected clustering features**
---
`CV_MJ/kg_db` for Energy density\
`VM_db` for Volatility → pyrolysis suitability\
`FC_db` for Char formation → combustion suitability\
`Ash_db` for Inert burden\
`Volatile_Fixed_Ratio` for Reaction behavior\
`Moisture_Penalty` for Practical usability

**Ash oxides are excluded** becasues they describe reactor-side risks(slagging/fouling),and not fuel-typology identity.

In [ ]:
import pandas as pd

import sys
import os
sys.path.append(os.path.abspath(".."))          # add parent file to your file search/look into parent file

from src.clustering.build_cluster_df import (
    build_clustering_df
)

features_df = pd.read_csv(                      
    "../data/interim/engineered_features.csv"
)

cluster_df = build_clustering_df(features_df)           
cluster_df.to_csv("../data/processed/clustering_ready.csv", index=False)

In [ ]:
from src.clustering.preprocess import scale_features                # scale features 
from src.clustering.evaluation import evaluate_k_range              # evaluate cluster quality by inertia and silhoutte

X_scaled, scaler = scale_features(cluster_df.drop(columns=["Sample_ID"]))       # Sample_ID is only a reference key,

scores = evaluate_k_range(X_scaled)                         # evaluation of different 'k' values on scaled data
scores

[{'k': 2, 'inertia': 483.4076387406313, 'silhouette': 0.6031550621732015},
 {'k': 3, 'inertia': 270.7941775245866, 'silhouette': 0.6219777580275656},
 {'k': 4, 'inertia': 199.48364748528834, 'silhouette': 0.2977991904915569},
 {'k': 5, 'inertia': 161.96088144922018, 'silhouette': 0.32986875787711295},
 {'k': 6, 'inertia': 139.7975686211198, 'silhouette': 0.35112698854650964},
 {'k': 7, 'inertia': 119.25109852893297, 'silhouette': 0.3663819895155893}]

**Decision of 'k' selection:** `k = 3` with inertia ≈ 270.79 (within-cluster compactness) and silhouette ≈ 0.621 (cluster separation quality): the elbow point,
after which cluster quality collapses.

**3 energy convsersion regimes:**

1. high energy/low ash/reactive fuels
2. moderate enrgy/mixed fuels\
3. low energy/high ash/constrained fuels

In [3]:
# Saving inertia and silhouette scores
pd.DataFrame(scores).to_csv(
    "../results/tables/clustering/silhouette_scores.csv", index=False
)

In [ ]:
# Run KMeans clustering model
from src.clustering.kmeans import run_kmeans, canonicalize_cluster_labels

kmeans_model, raw_labels = run_kmeans(X_scaled, n_clusters=3)       # applied on scaled dataset

labels = canonicalize_cluster_labels(raw_labels, cluster_df["Ash_db"])

cluster_df["Cluster"] = labels                  # add cluster IDs in cluster_df on the basis of assigned labels

In [ ]:
# create cluster summary
cluster_summary = (
    cluster_df
    .drop(columns=["Sample_ID"])      # dropping column Sample_ID 
    .groupby("Cluster")               # compute average per cluster  
    .mean()
)

# saving cluster_summary file
cluster_summary.to_csv(
    "../results/tables/clustering/cluster_summary.csv"
)

In [6]:
cluster_summary

,CV_MJ/kg_db,VM_db,FC_db,Ash_db,Volatile_Fixed_Ratio,Moisture_Penalty
Cluster,,,,,,
0,40.600000,97.800000,0.900000,1.300000,108.666667,0.312500
1,19.528972,80.573364,16.724486,2.705047,5.205381,0.069803
2,12.926000,52.870000,11.750000,35.380000,4.995222,0.051041


**Cluster interpretation**

| Cluster | Energy regime | Best suited for |
|---|---|---|
| 0 | Ultra-high-energy, volatile-rich | Gasification / combustion / CHP |
| 1 | Balanced, conversion-ready | Direct combustion / co-firing / fast pyrolysis |
| 2 | Low-energy, ash-constrained | Co-processing / specialized / pre-treatment routes |

Values are cluster-mean dry-basis properties (see `cluster_summary.csv`).


In [ ]:
# Validation against known labels
original_df = pd.read_csv("../data/interim/validated_data.csv")

cluster_df["Biomass_Type"] = original_df.loc[cluster_df.index, "Biomass_Type"]           # align original labels with clusters sample

cluster_biomass = pd.crosstab(
    cluster_df["Cluster"],
    cluster_df["Biomass_Type"],      # if not normalize = no. of agr, for, ind, urb samples in cluster 0,1,2
    normalize="index"                # normalize to create % values of samples typs in clusters 0,1,2 
)

cluster_biomass.to_csv("../results/tables/clustering/cluster_biomass_pct_crosstab.csv")

In [8]:
cluster_biomass

Biomass_Type,Agricultural,Forestry,Industrial Processing,Urban Waste
Cluster,,,,
0,0.000000,0.00000,0.000000,1.000000
1,0.018692,0.46729,0.383178,0.130841
2,0.400000,0.00000,0.300000,0.300000


**Result:** Fuel clusters do not align 1:1 with administrative biomass
classes which validates that unsupervised typology captures thermochemical
behavior the `Biomass_Type` label alone does not.


In [9]:
# saving cluster_df after adding label(cluster) in KMeans +  biomass type(in validation) 
cluster_df.to_csv(
    "../results/tables/clustering/clustered_with_labels.csv", index=False
)

#### **Visualization**
---

In [10]:
from src.clustering.preprocess import scale_features
from src.clustering.evaluation import evaluate_k_range
from src.visualization.clustering_plots import (
    plot_elbow,
    plot_silhouette_scores,
    plot_pca_clusters
)

from sklearn.cluster import KMeans
import pandas as pd

In [11]:
# Elbow plot
k_range = range(2, 8)

plot_elbow(
    X_scaled,
    k_range,
    save_path="../results/figures/elbow_plot.png"
)

# Silhouette plots
plot_silhouette_scores(
    X_scaled,
    k_range,
    save_path="../results/figures/silhouette_plot.png"
)

# PCA Visualization
plot_pca_clusters(
    X_scaled,
    cluster_df["Cluster"],
    save_path="../results/figures/pca_clusters.png"
)

In [12]:
cluster_df.head()

,Sample_ID,CV_MJ/kg_db,VM_db,FC_db,Ash_db,Volatile_Fixed_Ratio,Moisture_Penalty,Cluster,Biomass_Type
0,1,20.55,81.0,17.9,1.1,4.525140,0.023697,1,Industrial Processing
1,3,12.41,56.2,11.1,32.7,5.063063,0.050251,2,Agricultural
2,4,12.66,52.0,5.2,42.8,10.000000,0.109890,2,Urban Waste
3,5,14.14,64.2,9.6,26.2,6.687500,0.113636,2,Industrial Processing
4,6,18.28,77.9,20.2,1.9,3.856436,0.079365,1,Industrial Processing
